# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [29]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector
from tqdm.notebook import tqdm
import itertools

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [30]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, date_column = 'timestamp'):
        self.date_column = date_column
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_transform = X.copy()
        X_transform['hour'] = X_transform.timestamp.dt.hour
        X_transform['weekday'] = X_transform.timestamp.dt.weekday
        X_transform.drop(columns=['timestamp'], inplace=True)
        return X_transform

In [31]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column):
        self.target_column = target_column
        self.encoder = OneHotEncoder(sparse_output=False)

    def fit(self, X):
        cat_selector = make_column_selector(dtype_include=['object', 'category'])
        self.cat_columns = cat_selector(X)
        if (self.target_column) in self.cat_columns:
            self.cat_columns.remove(self.target_column)
        self.encoder.fit(X[self.cat_columns])
        return self
    
    def transform(self, X):
        X_encode = X.copy()
        encode = self.encoder.transform(X[self.cat_columns])
        encode_df = pd.DataFrame(encode, columns=self.encoder.get_feature_names_out())

        X_transformed = pd.concat([X_encode.drop(columns=self.cat_columns), encode_df], axis=1)
        return X_transformed

    

In [32]:
class TrainValidationTest():
    def __init__(self, X, y):
        self.X = X.copy()
        self.y = y.copy()
    
    def split(self):
        X_pemp, X_test, y_temp, y_test = train_test_split(self.X, self.y, test_size=0.2, random_state=21, stratify=self.y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_pemp, y_temp, test_size=0.2, random_state=21, stratify=y_temp)

        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

# Доделать tqdm

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

кастомный GridSearchCV для отображения прогресса обучения при помощи библиотеки tqdm

In [33]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids # это объекты GridSearchCV, которые надо обучать и получать лучшие параметры
        self.grid_dict = grid_dict # это словаь в котором указаны названия моделей в в порядке их передачи
        # ну то есть если первым передали svc, то { 0 : 'SVM'}
        self.best_results_table = []

    def choose(self, X_train, X_valid, y_train, y_valid):
        best_classifier_number = None
        best_valid_score = -1

        for i, grid in enumerate(self.grids):
            print(f'Estimator: {self.grid_dict[i]}')

            grid.fit(X_train, y_train)

            # вывод результатов
            print(f'Best params: {grid.best_params_}')
            print(f'Best traning accuracy: {grid.best_score_:.3f}')
            best_model = grid.best_estimator_
            valid_predict = best_model.predict(X_valid)
            valid_score = accuracy_score(y_valid, valid_predict)
            print(f'Validation set accuracy score for best params: {valid_score:.3f}')
            print('')

            # сохранение результатов для опитимизации работы метдов
            d = {
                'model' : self.grid_dict[i],
                'params' : grid.best_params_,
                'valid_score' : valid_score
            }
            self.best_results_table.append(d)

            #выбор лучшей модели на основе valid_score
            if valid_score > best_valid_score:
                best_valid_score = valid_score
                best_classifier_number = i
        
        print(f'Classifier with best validation set accuracy: {self.grid_dict[best_classifier_number]}')
     
    def best_results(self):
        if len(self.best_results_table) != 0:
            return pd.DataFrame(self.best_results_table)
        else:
            raise Exception("Сначала выполните метод choose()")

        

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [34]:
class Finalize():
    def __init__(self, estimator):
        self.model = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.model.fit(X_train, y_train)
        predict = self.model.predict(X_test)
        print(f'ACcuracy of the final model is {accuracy_score(y_test, predict)}')

    def save_model(self, file_path):
        joblib.dump(self.model, file_path)
        print('model was successfullu saved')

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

load data from the file

In [45]:
df = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])
df.head()

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


create and use pipline for the dataset

In [46]:
preprocessing = Pipeline([
    ('feature_actractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])

In [47]:
data = preprocessing.fit_transform(df)
data

,numTrials,hour,weekday,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


get train valid and test sets

In [48]:
tr_val_test = TrainValidationTest(data.drop(columns='weekday'), data.weekday)
X_train, X_valid, X_test, y_train, y_valid, y_test = tr_val_test.split()

Use ModelSelection

In [49]:
svc_param_grid = {
    'kernel':['linear', 'rbf', 'sigmoid'],
    'C' : [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma' : ['scale', 'auto'],
    'class_weight' : ['balanced', None],
    'random_state' : [21],
    'probability' : [True]
}
svc_model = SVC()
grid_svc = GridSearchCV(svc_model, svc_param_grid, n_jobs=-1, scoring='accuracy', cv=2)

In [50]:
tree_param_grid = {
    'max_depth' : range(1,50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini'],
    'random_state' : [21]
}
tree_model = DecisionTreeClassifier()
grid_tree = GridSearchCV(tree_model, tree_param_grid, scoring='accuracy', n_jobs=-1, cv=2)

In [51]:
forest_param_grid = {
    'n_estimators' : [5,10,50,100],
    'max_depth' : range(1,50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini'],
    'random_state' : [21]
}
forest_model = RandomForestClassifier()
grid_forest = GridSearchCV(forest_model, forest_param_grid, scoring='accuracy', n_jobs=-1, cv=2)

In [52]:
grids = [grid_svc, grid_tree, grid_forest]
dict_grid = {
    0 : 'SVM',
    1 : 'Decision Tree',
    2 : 'Random Forest'
}

In [53]:
mod_sel = ModelSelection(grids, dict_grid)
mod_sel.choose(X_train, X_valid, y_train, y_valid)

Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best traning accuracy: 0.773
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22, 'random_state': 21}
Best traning accuracy: 0.809
Validation set accuracy score for best params: 0.867

Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best traning accuracy: 0.855
Validation set accuracy score for best params: 0.907

Classifier with best validation set accuracy: Random Forest


In [ ]:
mod_sel.best_results()

,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.877778
1,Decision Tree,"{'class_weight': 'balanced', 'criterion': 'gin...",0.866667
2,Random Forest,"{'class_weight': None, 'criterion': 'entropy',...",0.907407


Use Finalaze and save model

In [ ]:
forest_model = RandomForestClassifier(criterion='entropy',
                                      max_depth=22,
                                      n_estimators=50,
                                      random_state=21)
finn = Finalize(forest_model)
finn.final_score(X_train, y_train, X_test, y_test)

ACcuracy of the final model is 0.9053254437869822


In [ ]:
finn.save_model('Random_Forest_Classifier_0_905325.sav')

model was successfullu saved


In [ ]:
model = joblib.load('Random_Forest_Classifier_0_905325.sav')

In [ ]:
finn_2 = Finalize(model)

In [ ]:
finn_2.final_score(X_train, y_train, X_test, y_test)

ACcuracy of the final model is 0.9053254437869822
